# Comparação entre os grupos (Cooperative waste pickers vs. Independent waste pickers)

In [1]:
import pandas as pd
from statsmodels.stats.weightstats import ttest_ind, CompareMeans, DescrStatsW
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
caminho = r'C:\Users\saulosgil\OneDrive\Documentos\health_outcomes_workers\data\df_analise.csv'
df = pd.read_csv(caminho)

df.head()

,idade,sexo,estado_civil,escolaridade,renda_mensal,cor_raca,tempo_trab_anos,funcao_trab,carga_horaria_horas_sem,outro_trab,...,Diuretics,Beta_blockers,ACE_inhibitors,Metformin,CCB,whoqol_fisico,whoqol_psicologico,whoqol_social,whoqol_ambiente,whoqol_n_validos
0,34,feminino,separado,médio completo,1 a 3 salários mínimos,preto (a),1.0,cooperado,40,sim,...,0,0,0,0,0,71.43,70.83,83.33,78.12,26
1,59,masculino,divorciado,médio completo,mais de 3 salários,pardo (a),2.0,cooperado,56,não,...,0,0,0,0,0,100.00,83.33,100.00,93.75,26
2,27,masculino,solteiro,superior completo,mais de 3 salários,pardo (a),1.0,cooperado,44,não,...,0,0,0,0,0,78.57,58.33,75.00,75.00,26
3,40,masculino,solteiro,médio completo,1 a 3 salários mínimos,preto (a),1.0,cooperado,45,não,...,0,0,0,0,0,78.57,95.83,100.00,78.12,26
4,23,masculino,solteiro,médio completo,mais de 3 salários,preto (a),2.0,cooperado,45,não,...,0,0,0,0,0,75.00,83.33,66.67,90.62,26


Sunstituição dos termos em português para inglês

In [3]:
df['funcao_trab'] = df['funcao_trab'].replace({'cooperado': 'Cooperative waste pickers'})
df['funcao_trab'] = df['funcao_trab'].replace({'individual': 'Independent waste pickers'})

# Teste T para amostras independentes para as variáveis:
    
        1. Peso;
        
        2. IMC;
        
        3. Circunferência da cintura;
        
        4. Relação Circunferência da cintura altura

In [4]:
variaveis = ['peso', 'imc', 'circun_abdome', 'cintura_altura_ratio']

resultados = []

for var in variaveis:
    grupo1 = df.loc[df['funcao_trab'] == 'Independent waste pickers', var].dropna()
    grupo2 = df.loc[df['funcao_trab'] == 'Cooperative waste pickers',  var].dropna()

    cm = CompareMeans(DescrStatsW(grupo1), DescrStatsW(grupo2))
    t, p, gl = ttest_ind(grupo1, grupo2, usevar='unequal')
    ic_inf, ic_sup = cm.tconfint_diff(alpha=0.05, usevar='unequal')
    delta = grupo1.mean() - grupo2.mean()

    # Saída no console
    # status = 'Há' if p < 0.05 else 'Não há'
    # print(f'{var}: {status} diferença estatisticamente significante')
    # print(f'  Δ: {delta:.1f} (IC 95%: {ic_inf:.1f} a {ic_sup:.1f}), P = {p:.4f}\n')

    # Guarda na tabela
    resultados.append({
        'variavel': var,
        'independent (média ± DP)': f'{grupo1.mean():.1f} ± {grupo1.std():.1f}',
        'cooperative (média ± DP)': f'{grupo2.mean():.1f} ± {grupo2.std():.1f}',
        'Δ (IC 95%)': f'{delta:.1f} ({ic_inf:.1f} a {ic_sup:.1f})',
        't': round(t, 2),
        'gl': round(gl, 1),
        'P': round(p, 4),
    })

tabela_t = pd.DataFrame(resultados)
tabela_t

,variavel,independent (média ± DP),cooperative (média ± DP),Δ (IC 95%),t,gl,P
0,peso,66.0 ± 12.4,79.8 ± 16.4,-13.8 (-22.6 a -5.0),-3.17,40.5,0.0029
1,imc,25.9 ± 5.4,29.5 ± 5.8,-3.6 (-7.1 a -0.1),-2.08,35.8,0.0444
2,circun_abdome,83.6 ± 13.5,95.5 ± 14.0,-11.9 (-20.5 a -3.3),-2.81,35.0,0.0081
3,cintura_altura_ratio,52.6 ± 9.9,58.2 ± 9.1,-5.7 (-11.7 a 0.4),-1.90,31.9,0.0659


# BoxPlot das variáveis:
    
        1. Peso;
        
        2. IMC;
        
        3. Circunferência da cintura;
        
        4. Relação Circunferência da cintura altura

In [5]:
sns.set_theme(
    style='whitegrid',
    rc={
        'figure.dpi': 300,             # alta resolução (dentro do rc para não ser sobrescrito)
        'savefig.dpi': 300,
        'axes.facecolor': 'white',
        'figure.facecolor': 'white',
        'axes.edgecolor': 'black',
        'axes.linewidth': 1,
        'grid.color': 'black',
        'grid.alpha': 0.2,
        'xtick.color': 'black',
        'ytick.color': 'black',
        'xtick.bottom': True,
        'ytick.left': True,
        'axes.labelcolor': 'black',
        'text.color': 'black',
    },
)

variaveis = {
    'peso':                 'Weight (kg)',
    'imc':                  'BMI (kg/m²)',
    'circun_abdome':        'Waist circunference (cm)',
    'cintura_altura_ratio': 'Waist-to-Height Ratio (a.u)'
}

for var, rotulo in variaveis.items():
    fig, ax = plt.subplots()

    sns.boxplot(
        x=df['funcao_trab'],
        y=df[var],
        hue=df['funcao_trab'],
        width=.3,
        linewidth=2,
        legend=False,
        ax=ax,
    )

    ax.set_xlabel('')
    ax.set_ylabel(rotulo)

    plt.tight_layout()
    plt.savefig(f'figura_1/boxplot_{var}.png', format='png')      # salvar ANTES do show
    # plt.savefig(f'boxplot_{var}.svg', format='svg')    # vetorial
    plt.close(fig)

# Teste de Fisher para amostras independentes para as variáveis:
    
        1. obesidade;
        
        2. aumentado_cmr;

In [6]:
import pandas as pd
from scipy.stats import fisher_exact
from scipy.stats.contingency import odds_ratio

variaveis_bin = ['obesidade', 'aumentado_cmr']

g1 = 'Independent waste pickers'
g2 = 'Cooperative waste pickers'

resultados_fisher = []

for var in variaveis_bin:
    dados = df[['funcao_trab', var]].dropna()

    # Tabela 2x2: linhas = grupos (g1, g2); colunas = condição (1, 0)
    tab = (
        pd.crosstab(dados['funcao_trab'], dados[var])
        .reindex(index=[g1, g2], columns=[1, 0], fill_value=0)
    )

    _, p = fisher_exact(tab.values, alternative='two-sided')
    or_res = odds_ratio(tab.values)                       # OR condicional (mesma base do Fisher)
    or_val = or_res.statistic
    ic = or_res.confidence_interval(confidence_level=0.95)

    n1, n2 = tab.loc[g1].sum(), tab.loc[g2].sum()
    pct1 = tab.loc[g1, 1] / n1 * 100
    pct2 = tab.loc[g2, 1] / n2 * 100
    delta = pct1 - pct2

    # Saída no console
    status = 'Há' if p < 0.05 else 'Não há'
    print(f'{var}: {status} diferença estatisticamente significante')
    print(f'  Δ: {delta:.1f} p.p., OR: {or_val:.2f} (IC 95%: {ic.low:.2f} a {ic.high:.2f}), P = {p:.4f}\n')

    # Guarda na tabela
    resultados_fisher.append({
        'variavel': var,
        'independent, n (%)': f'{tab.loc[g1, 1]} ({pct1:.0f})',
        'cooperative, n (%)': f'{tab.loc[g2, 1]} ({pct2:.0f})',
        'Δ (p.p.)': round(delta, 1),
        'OR (IC 95%)': f'{or_val:.2f} ({ic.low:.2f} a {ic.high:.2f})',
        'P': round(p, 4),
    })

tabela_fisher = pd.DataFrame(resultados_fisher)
tabela_fisher

obesidade: Não há diferença estatisticamente significante
  Δ: -26.1 p.p., OR: 0.34 (IC 95%: 0.07 a 1.41), P = 0.1244

aumentado_cmr: Não há diferença estatisticamente significante
  Δ: -22.7 p.p., OR: 0.33 (IC 95%: 0.07 a 1.57), P = 0.1641



,variavel,"independent, n (%)","cooperative, n (%)",Δ (p.p.),OR (IC 95%),P
0,obesidade,5 (29),15 (56),-26.1,0.34 (0.07 a 1.41),0.1244
1,aumentado_cmr,10 (59),22 (81),-22.7,0.33 (0.07 a 1.57),0.1641


# Gráfico de barras (frequencia relativa) para as variáveis:
    
        1. obesidade;
        
        2. aumentado_cmr;

In [7]:
sns.set_theme(
    style='whitegrid',
    rc={
        'figure.dpi': 300,
        'savefig.dpi': 300,
        'axes.facecolor': 'white',
        'figure.facecolor': 'white',
        'axes.edgecolor': 'black',
        'axes.linewidth': 1,
        'grid.color': 'black',
        'grid.alpha': 0.2,
        'xtick.color': 'black',
        'ytick.color': 'black',
        'xtick.bottom': True,
        'ytick.left': True,
        'axes.labelcolor': 'black',
        'text.color': 'black',
    },
)

variaveis_bin = {
    'obesidade':     'Obesity (%)',
    'aumentado_cmr': 'Increased cardiometabolic risk (%)',
}

for var, rotulo in variaveis_bin.items():
    # % de participantes com a condição (valor = 1) em cada grupo
    freq = (
        df.groupby('funcao_trab')[var]
        .mean()
        .mul(100)
        .reset_index(name='percentual')
    )

    fig, ax = plt.subplots()

    sns.barplot(
        data=freq,
        x='funcao_trab',
        y='percentual',
        hue='funcao_trab',
        width=.3,
        linewidth=2,
        edgecolor='black',
        legend=False,
        ax=ax,
    )

    # Rótulo de % em cima de cada barra
    for container in ax.containers:
        ax.bar_label(container, fmt='%.0f%%', padding=3)

    ax.set_ylim(0, 100)
    ax.set_xlabel('')
    ax.set_ylabel(rotulo)

    plt.tight_layout()
    plt.savefig(f'figura_1/barplot_{var}.png', format='png')
    # plt.savefig(f'figura_1/barplot_{var}.svg', format='svg')
    plt.close(fig)